# 04 - Eye movements

**Audience:** running the I-VT classifier and using the output.

Default thresholds match the I-VT literature (50 deg/s velocity, 100 ms
minimum fixation, 75 ms merge window). The first section sweeps the
velocity threshold so you can pick a value justified for your data.

## What you'll get

- Velocity-threshold sweep (visual signature of the elbow).
- Fixation/saccade summary statistics.
- Scanpath plot with fixation centroids sized by duration.
- K-coefficient attention typing (focal vs. ambient).

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Convert combined gaze to angular yaw/pitch

In [ ]:
df = ctx.data.df
ts = ctx.data.get_timestamps()
for c in ("combined_dir_x", "combined_dir_y", "combined_dir_z"):
    if c not in df.columns:
        raise RuntimeError(f"Missing required column: {c}")
dx = df["combined_dir_x"].to_numpy(dtype=float)
dy = df["combined_dir_y"].to_numpy(dtype=float)
dz = df["combined_dir_z"].to_numpy(dtype=float)
yaw = np.degrees(np.arctan2(dx, dz))
pitch = -np.degrees(np.arcsin(np.clip(dy, -1, 1)))

## 2. Velocity-threshold sweep

Vary the I-VT threshold and watch fixation count + median duration. The
literature 50 deg/s sits in the elbow of this curve for typical VR data.

In [ ]:
thresholds = [20, 30, 40, 50, 75, 100, 150]
rows = []
for th in thresholds:
    mv = ela.detect_eye_movements(yaw, pitch, ts, velocity_threshold=th)
    fixs = mv["fixations"]
    durs_ms = [f.duration * 1000 for f in fixs] if fixs else [0]
    rows.append({
        "threshold_deg_s": th,
        "n_fixations": len(fixs),
        "n_saccades": len(mv["saccades"]),
        "median_fix_ms": np.median(durs_ms),
        "p95_fix_ms": np.percentile(durs_ms, 95),
    })
sweep = pd.DataFrame(rows)
sweep

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(sweep["threshold_deg_s"], sweep["n_fixations"], marker="o")
axes[0].set_xlabel("velocity threshold (deg/s)")
axes[0].set_ylabel("# fixations")
axes[0].set_title("Fixation count vs. threshold")
axes[1].plot(sweep["threshold_deg_s"], sweep["median_fix_ms"], marker="o", color="darkorange")
axes[1].set_xlabel("velocity threshold (deg/s)")
axes[1].set_ylabel("median duration (ms)")
axes[1].set_title("Median fixation duration")
plt.tight_layout()
plt.show()

## 3. Detection at the chosen threshold (50 deg/s default)

In [ ]:
movements = ela.detect_eye_movements(yaw, pitch, ts, velocity_threshold=50.0)
fixs = movements["fixations"]
saccs = movements["saccades"]
if fixs:
    durs_ms = np.array([f.duration * 1000 for f in fixs])
    print(f"Fixations:  n={len(fixs)}  median={np.median(durs_ms):.1f} ms  p95={np.percentile(durs_ms, 95):.1f}")
if saccs:
    amps = np.array([s.amplitude for s in saccs])
    pks = np.array([s.peak_velocity for s in saccs])
    print(f"Saccades:   n={len(saccs)}  median amp={np.median(amps):.2f} deg  median peak vel={np.median(pks):.0f} deg/s")

## 4. Scanpath

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(yaw, pitch, color="lightgray", linewidth=0.4, label="raw")
if fixs:
    fx = [f.centroid_x for f in fixs]
    fy = [f.centroid_y for f in fixs]
    sizes = [max(20, min(400, f.duration * 1000)) for f in fixs]
    ax.scatter(fx, fy, s=sizes, alpha=0.6, edgecolor="black", label="fixation")
ax.set_xlabel("yaw (deg)")
ax.set_ylabel("pitch (deg)")
ax.invert_yaxis()
ax.set_aspect("equal", adjustable="datalim")
ax.set_title(f"Scanpath - {ctx.csv_path.name}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. K-coefficient (Krejtz et al. 2016)

Positive K = focal attention (long fixations, short saccades).
Negative K = ambient attention (short fixations, long saccades).

In [ ]:
if len(fixs) >= 2 and len(saccs) >= 1:
    k = ela.calculate_k_coefficient(fixs, saccs)
    print(f"K = {k.k_coefficient:+.4f}")
    print(f"Attention type: {k.attention_type.value if hasattr(k.attention_type, 'value') else k.attention_type}")
    print(f"  mean fixation: {k.mean_fixation_duration*1000:.1f} ms")
    print(f"  mean saccade:  {k.mean_saccade_amplitude:.2f} deg")
else:
    print("Need at least 2 fixations and 1 saccade for K-coefficient.")